# Gettel Title Organizer

This notebook organizes your downloaded Gettel PDFs into two folders on Google Drive:
- **Property Sheets/** — renamed from `Property XXXXX.pdf` → `XXXXX.pdf`
- **Title Documents/** — renamed from site filename → `PID_filename.pdf`

### Before running:
1. Open [Google Drive](https://drive.google.com) in another tab
2. Create a folder called **`Gettel`** (or any name you like)
3. Upload all your downloaded PDFs into that folder
4. Upload the CSV exported by the scraper into that same folder
5. Then run the cells below top to bottom (Shift+Enter or click the ▶ button)

## Step 1 — Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected.')

## Step 2 — Set your folder path

Change `Gettel` below to whatever you named your folder on Drive.
If it's inside another folder, use the full path, e.g. `My Drive/Real Estate/Gettel`.

In [ ]:
import os

# ── EDIT THIS LINE ────────────────────────────────────────────────────────────
FOLDER = 'Gettel'
# ─────────────────────────────────────────────────────────────────────────────

BASE = f'/content/drive/My Drive/{FOLDER}'

if not os.path.isdir(BASE):
    raise FileNotFoundError(f'Folder not found: {BASE}\nCheck the name matches exactly (case-sensitive).')

print(f'Folder found: {BASE}')
print(f'Files inside: {len(os.listdir(BASE))}')

## Step 3 — Pick the CSV file

Run this cell — it will list all CSV files it finds in your folder so you can confirm the right one.

In [ ]:
csvs = [f for f in os.listdir(BASE) if f.lower().endswith('.csv')]

if not csvs:
    raise FileNotFoundError('No CSV files found in the folder. Make sure you uploaded the scraper export.')

print('CSV files found:')
for i, name in enumerate(csvs):
    print(f'  [{i}] {name}')

# If there is only one CSV it is selected automatically.
# If there are multiple, change the number below to match the one you want.
CSV_INDEX = 0
CSV_PATH  = os.path.join(BASE, csvs[CSV_INDEX])
print(f'\nUsing: {csvs[CSV_INDEX]}')

## Step 4 — Organize the files

Run this cell to rename and sort everything. It will print a log of every file moved.

In [ ]:
import csv
import shutil

PROP_FOLDER  = os.path.join(BASE, 'Property Sheets')
TITLE_FOLDER = os.path.join(BASE, 'Title Documents')
os.makedirs(PROP_FOLDER,  exist_ok=True)
os.makedirs(TITLE_FOLDER, exist_ok=True)

def find_file(folder, name):
    """Case-insensitive file lookup."""
    target = name.lower()
    for f in os.listdir(folder):
        if f.lower() == target:
            return os.path.join(folder, f)
    return None

moved, skipped, not_found = 0, 0, 0

with open(CSV_PATH, newline='', encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))

print(f'Processing {len(rows)} rows from CSV...\n')

for row in rows:
    pid      = (row.get('PID')      or '').strip()
    status   = (row.get('Status')   or '').strip().lower()
    filename = (row.get('Filename') or '').strip()

    if not pid:
        continue

    # ── Property sheet ────────────────────────────────────────────────────────
    prop_src = find_file(BASE, f'Property {pid}.pdf')
    if prop_src:
        dest = os.path.join(PROP_FOLDER, f'{pid}.pdf')
        if os.path.exists(dest):
            print(f'[SKIP]  Property {pid}.pdf → already exists')
            skipped += 1
        else:
            shutil.move(prop_src, dest)
            print(f'[OK]    Property {pid}.pdf → Property Sheets/{pid}.pdf')
            moved += 1

    # ── Title / LINC ──────────────────────────────────────────────────────────
    if status == 'downloaded' and filename:
        title_src = find_file(BASE, filename)
        if title_src:
            new_name = f'{pid}_{filename}' if not filename.startswith(f'{pid}_') else filename
            dest     = os.path.join(TITLE_FOLDER, new_name)
            if os.path.exists(dest):
                print(f'[SKIP]  {filename} → already exists as {new_name}')
                skipped += 1
            else:
                shutil.move(title_src, dest)
                print(f'[OK]    {filename} → Title Documents/{new_name}')
                moved += 1
        else:
            print(f'[MISS]  {filename} (PID {pid}) — not found in folder')
            not_found += 1

print(f'\n{"-"*50}')
print(f'Done.  Moved: {moved}  |  Already existed: {skipped}  |  Not found: {not_found}')
if not_found:
    print('Files marked [MISS] were not in the folder — check they uploaded correctly.')